# 05 - ImageNet-100 End-to-End Training

Joint fine-tuning of VAE Healer + ResNet Expert:
- Load pretrained models from notebooks 02 and 03
- Combined loss: VAE reconstruction + Classification
- Gradient accumulation for memory efficiency
- Save fine-tuned models

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os

# Configuration
BASE_DIR = r"C:\Users\Rushikesh\OneDrive\CODES\SelfHealingNN"
os.chdir(BASE_DIR)

torch.set_num_threads(8)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 224
BATCH_SIZE = 16  # Reduced for joint training
ACCUMULATION_STEPS = 2  # Effective batch = 32
NUM_CLASSES = 100

print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Define Models

In [ ]:
class ImageNet100VAE(nn.Module):
    def __init__(self, latent_dim=256):
        super(ImageNet100VAE, self).__init__()
        self.latent_dim = latent_dim
        
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 3, stride=2, padding=1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
        )
        
        self.flatten_size = 512 * 7 * 7
        self.fc_mu = nn.Linear(self.flatten_size, latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_size, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, self.flatten_size)
        
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid(),
        )
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std
    
    def forward(self, x):
        h = self.encoder(x).view(-1, self.flatten_size)
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        h2 = F.relu(self.fc_decode(z)).view(-1, 512, 7, 7)
        return self.decoder(h2), mu, logvar


class ImageNet100Expert(nn.Module):
    def __init__(self, num_classes=100, dropout=0.3):
        super(ImageNet100Expert, self).__init__()
        self.resnet = models.resnet18(weights=None)
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes)
        )
    
    def forward(self, x):
        return self.resnet(x)

print("Model architectures defined")

## 2. Load Pretrained Models

In [ ]:
# Load pretrained models
healer = ImageNet100VAE(latent_dim=256).to(device)
healer.load_state_dict(torch.load(os.path.join(BASE_DIR, "models", "imagenet100_healer.pth")))

expert = ImageNet100Expert(num_classes=NUM_CLASSES).to(device)
expert.load_state_dict(torch.load(os.path.join(BASE_DIR, "models", "imagenet100_expert.pth")))

healer_params = sum(p.numel() for p in healer.parameters())
expert_params = sum(p.numel() for p in expert.parameters())

print(f"Models loaded:")
print(f"  Healer: {healer_params:,} parameters")
print(f"  Expert: {expert_params:,} parameters")
print(f"  Total:  {healer_params + expert_params:,} parameters")

## 3. Load Data

In [ ]:
# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Transforms WITHOUT normalization (for VAE pipeline)
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

# Data paths
DATA_DIR = os.path.join(BASE_DIR, "imagenet100_data")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")

# Datasets
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataset):,} images ({len(train_loader)} batches)")
print(f"Val: {len(val_dataset):,} images")

## 4. Helper Functions

In [ ]:
def add_noise(image_tensor, noise_factor=0.4):
    """Add Gaussian noise."""
    noise = torch.randn_like(image_tensor) * noise_factor
    return torch.clamp(image_tensor + noise, 0., 1.)


def normalize_for_resnet(tensor):
    """Normalize for ResNet."""
    mean = torch.tensor(IMAGENET_MEAN, device=tensor.device).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=tensor.device).view(1, 3, 1, 1)
    return (tensor - mean) / std


def vae_loss(recon_x, x, mu, logvar, beta=0.5):
    """VAE loss."""
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + beta * KLD


print("Helper functions defined")

## 5. End-to-End Training Configuration

In [ ]:
# Training configuration
EPOCHS = 20
LEARNING_RATE = 5e-5  # Low LR for fine-tuning
NOISE_FACTOR = 0.4
BETA = 0.5  # KLD weight
LAMBDA_CLS = 3.0  # Classification loss weight

# Joint optimizer
all_params = list(healer.parameters()) + list(expert.parameters())
optimizer = optim.Adam(all_params, lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)

# Classification loss with label smoothing
cls_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print(f"Training configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE} (effective: {BATCH_SIZE * ACCUMULATION_STEPS})")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Noise factor: {NOISE_FACTOR}")
print(f"  Lambda (cls weight): {LAMBDA_CLS}")
print(f"  Gradient accumulation: {ACCUMULATION_STEPS} steps")

## 6. Training Loop

In [ ]:
def evaluate_e2e(healer, expert, loader, noise_factor, device):
    """Evaluate end-to-end pipeline."""
    healer.eval()
    expert.eval()
    
    correct_clean = 0
    correct_noisy = 0
    correct_healed = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            total += labels.size(0)
            
            # Clean
            clean_norm = normalize_for_resnet(images)
            _, clean_pred = expert(clean_norm).max(1)
            correct_clean += clean_pred.eq(labels).sum().item()
            
            # Noisy
            noisy = add_noise(images, noise_factor)
            noisy_norm = normalize_for_resnet(noisy)
            _, noisy_pred = expert(noisy_norm).max(1)
            correct_noisy += noisy_pred.eq(labels).sum().item()
            
            # Healed
            healed, _, _ = healer(noisy)
            healed_norm = normalize_for_resnet(healed)
            _, healed_pred = expert(healed_norm).max(1)
            correct_healed += healed_pred.eq(labels).sum().item()
    
    return {
        'clean': 100. * correct_clean / total,
        'noisy': 100. * correct_noisy / total,
        'healed': 100. * correct_healed / total,
    }

print("Evaluation function ready")

In [ ]:
print(f"\nStarting End-to-End Training...")
print("="*70)

train_losses = []
metrics_history = []
best_healed_acc = 0.0

for epoch in range(EPOCHS):
    healer.train()
    expert.train()
    
    epoch_loss = 0.0
    epoch_vae_loss = 0.0
    epoch_cls_loss = 0.0
    optimizer.zero_grad()
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        # Add noise
        noisy = add_noise(images, NOISE_FACTOR)
        
        # Forward through healer
        healed, mu, logvar = healer(noisy)
        
        # Forward through expert
        healed_norm = normalize_for_resnet(healed)
        predictions = expert(healed_norm)
        
        # Combined loss
        loss_vae = vae_loss(healed, images, mu, logvar, beta=BETA)
        loss_cls = cls_criterion(predictions, labels)
        total_loss = loss_vae + LAMBDA_CLS * loss_cls
        
        # Scale for gradient accumulation
        scaled_loss = total_loss / ACCUMULATION_STEPS
        scaled_loss.backward()
        
        epoch_loss += total_loss.item()
        epoch_vae_loss += loss_vae.item()
        epoch_cls_loss += loss_cls.item()
        
        # Update weights every ACCUMULATION_STEPS
        if (batch_idx + 1) % ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(all_params, 1.0)
            optimizer.step()
            optimizer.zero_grad()
        
        # Progress
        if batch_idx % 200 == 0:
            avg_loss = epoch_loss / (batch_idx + 1)
            print(f"  Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx:>4}/{len(train_loader)} | Loss: {avg_loss:,.1f}")
    
    scheduler.step()
    
    # Evaluate
    metrics = evaluate_e2e(healer, expert, val_loader, NOISE_FACTOR, device)
    recovery = metrics['healed'] - metrics['noisy']
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    metrics_history.append(metrics)
    lr = optimizer.param_groups[0]['lr']
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:,.1f} | LR: {lr:.1e}")
    print(f"  Clean: {metrics['clean']:.1f}% | Noisy: {metrics['noisy']:.1f}% | Healed: {metrics['healed']:.1f}% | Recovery: +{recovery:.1f}%")
    
    # Save best model
    if metrics['healed'] > best_healed_acc:
        best_healed_acc = metrics['healed']
        torch.save(healer.state_dict(), os.path.join(BASE_DIR, "models", "imagenet100_healer_finetuned.pth"))
        torch.save(expert.state_dict(), os.path.join(BASE_DIR, "models", "imagenet100_expert_finetuned.pth"))
        print(f"  Saved best model (healed acc: {best_healed_acc:.1f}%)")
    
    print("-"*70)

print("\nEnd-to-End training complete!")

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(train_losses, 'b-', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('End-to-End Training Loss')
axes[0].grid(True, alpha=0.3)

# Accuracy curves
clean_acc = [m['clean'] for m in metrics_history]
noisy_acc = [m['noisy'] for m in metrics_history]
healed_acc = [m['healed'] for m in metrics_history]

axes[1].plot(clean_acc, 'g-', label='Clean', linewidth=2)
axes[1].plot(noisy_acc, 'r-', label='Noisy', linewidth=2)
axes[1].plot(healed_acc, 'b-', label='Healed', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Validation Accuracy During Fine-tuning')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Final Evaluation

In [ ]:
# Load best models
healer.load_state_dict(torch.load(os.path.join(BASE_DIR, "models", "imagenet100_healer_finetuned.pth")))
expert.load_state_dict(torch.load(os.path.join(BASE_DIR, "models", "imagenet100_expert_finetuned.pth")))

# Final evaluation at different noise levels
noise_levels = [0.2, 0.3, 0.4, 0.5]
final_results = []

print("Final Evaluation (Fine-tuned Models):")
print("="*60)

for noise in noise_levels:
    metrics = evaluate_e2e(healer, expert, val_loader, noise, device)
    recovery = metrics['healed'] - metrics['noisy']
    
    print(f"Noise {noise}: Clean={metrics['clean']:.1f}% | Noisy={metrics['noisy']:.1f}% | Healed={metrics['healed']:.1f}% | Recovery=+{recovery:.1f}%")
    
    final_results.append({
        'noise': noise,
        'clean': metrics['clean'],
        'noisy': metrics['noisy'],
        'healed': metrics['healed'],
        'recovery': recovery,
    })

print("="*60)

## 9. Visualize Healing Quality

In [ ]:
# Get sample images
images, labels = next(iter(val_loader))
images = images.to(device)

noise_factor = 0.4
noisy = add_noise(images, noise_factor)

with torch.no_grad():
    healed, _, _ = healer(noisy)

# Visualize
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
class_names = val_dataset.classes

for col in range(6):
    for row_idx, (img, title) in enumerate([
        (images[col], "Original"),
        (noisy[col], "Noisy"),
        (healed[col], "Healed")
    ]):
        img_np = img.cpu().numpy().transpose(1, 2, 0)
        axes[row_idx, col].imshow(np.clip(img_np, 0, 1))
        axes[row_idx, col].axis('off')
        if col == 0:
            axes[row_idx, col].set_ylabel(title, fontsize=12, rotation=0, labelpad=50)
        if row_idx == 0:
            axes[row_idx, col].set_title(class_names[labels[col]][:15], fontsize=10)

plt.suptitle(f"Fine-tuned Self-Healing Pipeline (Noise: {noise_factor})", fontsize=14)
plt.tight_layout()
plt.show()

## 10. Final Summary

In [ ]:
# Best result at noise 0.4
best_result = final_results[2]  # noise 0.4

print("="*70)
print("SELF-HEALING NEURAL NETWORK - FINAL RESULTS")
print("="*70)
print(f"")
print(f"Dataset: ImageNet-100")
print(f"  - 130,000 training images")
print(f"  - 100 diverse classes")
print(f"  - 224x224 RGB resolution")
print(f"")
print(f"Architecture:")
print(f"  - Healer: VAE (5-block encoder/decoder, ~7.5M params)")
print(f"  - Expert: ResNet-18 (pretrained, ~11.2M params)")
print(f"")
print(f"Key Results (Noise={best_result['noise']}):")
print(f"  Clean accuracy:  {best_result['clean']:.1f}%")
print(f"  Noisy accuracy:  {best_result['noisy']:.1f}%")
print(f"  Healed accuracy: {best_result['healed']:.1f}%")
print(f"  Recovery:        +{best_result['recovery']:.1f}%")
print(f"")
print(f"Models saved:")
print(f"  - models/imagenet100_healer_finetuned.pth")
print(f"  - models/imagenet100_expert_finetuned.pth")
print(f"")
print("="*70)
print("Project Complete!")
print("="*70)